# nanochat d8 chat SFT

This notebook is designed to run the **SFT-only** part of the `nanochat` pipeline on **Kaggle**.
The d8 base-model pretraining stage should already be finished and attached as a Kaggle Dataset.

## Before you run it

Make sure the notebook is configured with:

- **GPU enabled** (`2x Tesla T4` expected)
- **Internet enabled**
- Your uploaded Kaggle dataset **`nanochat-codes`** attached
- Your saved pretraining cache dataset **`nanochat-d8-pretrain-cache`** attached

The notebook auto-detects datasets from the mounted Kaggle path pattern
`/kaggle/input/datasets/<your-kaggle-username>/<dataset-name>`.

The `nanochat-codes` dataset is expected to contain the same code that lives under
`kaggle_dataset/nanochat/`. The saved pretraining cache is expected to contain
`base_checkpoints/d8_kaggle` and the matching `tokenizer` directory.

## What this notebook does

This notebook runs the SFT path only:

1. Kaggle environment setup and dependency installation
2. Download identity conversations for chat SFT
3. Import the saved `base_checkpoints/d8_kaggle` and matching tokenizer
4. Run supervised fine-tuning on top of the saved d8 base model
5. Evaluate and serve the SFT checkpoint


In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

DATASETS_ROOT = Path('/kaggle/input/datasets')
dataset_candidates = sorted(DATASETS_ROOT.glob('*/nanochat-codes'))

# On Kaggle, this notebook expects an attached dataset named 'nanochat-codes'.
# In this repo, the contents of that mounted dataset folder correspond to the
# local folder 'kaggle_dataset/nanochat/'. The outer 'kaggle_dataset/'
# directory is only the packaging wrapper used with the Kaggle CLI.
assert dataset_candidates, (
    "Could not find an attached Kaggle dataset matching "
    "'/kaggle/input/datasets/<username>/nanochat-codes'"
)
assert len(dataset_candidates) == 1, (
    f"Expected exactly one attached 'nanochat-codes' dataset, found: {dataset_candidates}"
)
CODE_INPUT = dataset_candidates[0]

assert CODE_INPUT.exists(), f'Code dataset not found: {CODE_INPUT}'
print('Code input:', CODE_INPUT)
print('Top-level code files:', sorted(p.name for p in CODE_INPUT.iterdir()))


Code input: /kaggle/input/datasets/yshuaiqin/nanochat-codes
Top-level code files: ['nanochat', 'pyproject.toml', 'scripts', 'tasks', 'tests']


In [2]:
WORK_REPO = Path('/kaggle/working/nanochat')
WORK_CACHE = Path('/kaggle/working/nanochat_cache')
HF_CACHE = Path('/kaggle/working/huggingface_cache')

for path in [WORK_REPO, WORK_CACHE, HF_CACHE]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

shutil.copytree(CODE_INPUT, WORK_REPO, dirs_exist_ok=True)

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['NANOCHAT_BASE_DIR'] = str(WORK_CACHE)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HF_DATASETS_CACHE'] = str(HF_CACHE / 'datasets')
os.environ['NANOCHAT_DTYPE'] = 'float16'
os.environ['PYTHONFAULTHANDLER'] = '1'


print('Working repo:', WORK_REPO)
print('Nanochat base dir:', WORK_CACHE)
print('HF cache:', HF_CACHE)
print('Repo contents:', sorted(p.name for p in WORK_REPO.iterdir()))
print('NANOCHAT_DTYPE:', os.environ['NANOCHAT_DTYPE'])


Working repo: /kaggle/working/nanochat
Nanochat base dir: /kaggle/working/nanochat_cache
HF cache: /kaggle/working/huggingface_cache
Repo contents: ['nanochat', 'pyproject.toml', 'scripts', 'tasks', 'tests']
NANOCHAT_DTYPE: float16


In [3]:
# Input HuggingFace token
import getpass

os.environ["HF_TOKEN"] = getpass.getpass("HF_TOKEN: ")

HF_TOKEN:  ········


In [4]:
packages = [
    'datasets>=4.0.0',
    'filelock>=3.0.0',
    'psutil>=7.1.0',
    'python-dotenv>=1.2.1',
    'pyyaml>=6.0.0',
    'regex>=2025.9.1',
    'requests>=2.32.0',
    'rustbpe>=0.1.0',
    'scipy>=1.15.3',
    'tabulate>=0.9.0',
    'tiktoken>=0.11.0',
    'tokenizers>=0.22.0',
    'transformers>=4.57.3',
    'wandb>=0.21.3',
    'zstandard>=0.25.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.9 MB/s eta 0:00:00


0

In [5]:
import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA available: True
CUDA device count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## 1. Download Identity Conversations for Chat SFT

This small JSONL file gives the model a lightweight assistant identity before chat fine-tuning.


In [6]:
cmd = [
    'curl',
    '-L',
    '-o', str(WORK_CACHE / 'identity_conversations.jsonl'),
    'https://karpathy-public.s3.us-west-2.amazonaws.com/identity_conversations.jsonl',
]

env = os.environ.copy()
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)
print('Saved:', WORK_CACHE / 'identity_conversations.jsonl')


Running: curl -L -o /kaggle/working/nanochat_cache/identity_conversations.jsonl https://karpathy-public.s3.us-west-2.amazonaws.com/identity_conversations.jsonl


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

Saved: /kaggle/working/nanochat_cache/identity_conversations.jsonl


100 2452k  100 2452k    0     0  3518k      0 --:--:-- --:--:-- --:--:-- 3519k


## 2. Import Saved d8 Base Checkpoints

This SFT notebook does not run pretraining. It imports the already-saved d8 base checkpoint from the attached Kaggle Dataset and places it under `WORK_CACHE/base_checkpoints`, where `scripts.chat_sft_updated` expects to find `base:d8_kaggle`.

The matching tokenizer is copied from the same saved cache because model loading checks tokenizer compatibility.


In [7]:
BASE_CACHE_DATASET = 'nanochat-d8-pretrain-cache'
MODEL_TAG = 'd8_kaggle'

cache_candidates = sorted(DATASETS_ROOT.glob(f'*/{BASE_CACHE_DATASET}'))
assert cache_candidates, (
    'Could not find an attached Kaggle dataset matching '
    f"'/kaggle/input/datasets/<username>/{BASE_CACHE_DATASET}'"
)
assert len(cache_candidates) == 1, (
    f"Expected exactly one attached '{BASE_CACHE_DATASET}' dataset, found: {cache_candidates}"
)
BASE_CACHE_INPUT = cache_candidates[0]

required_cache_dirs = ['base_checkpoints', 'tokenizer']
for dirname in required_cache_dirs:
    source = BASE_CACHE_INPUT / dirname
    destination = WORK_CACHE / dirname
    assert source.exists(), f'Missing {dirname} in saved pretraining cache: {source}'
    if destination.exists():
        shutil.rmtree(destination)
    shutil.copytree(source, destination)

base_dir = WORK_CACHE / 'base_checkpoints' / MODEL_TAG
assert base_dir.exists(), f'Missing imported base checkpoint directory: {base_dir}'
model_files = sorted(base_dir.glob('model_*.pt'))
meta_files = sorted(base_dir.glob('meta_*.json'))
assert model_files, f'No model_*.pt files found in {base_dir}'
assert meta_files, f'No meta_*.json files found in {base_dir}'

print('Imported saved cache:', BASE_CACHE_INPUT)
print('Imported base checkpoint:', base_dir)
print('Latest model file:', model_files[-1].name)
print('Tokenizer files:', sorted(p.name for p in (WORK_CACHE / 'tokenizer').iterdir()))


Imported saved cache: /kaggle/input/datasets/yshuaiqin/nanochat-d8-pretrain-cache
Imported base checkpoint: /kaggle/working/nanochat_cache/base_checkpoints/d8_kaggle
Latest model file: model_001920.pt
Tokenizer files: ['token_bytes.pt', 'tokenizer.pkl']


## 3. Chat SFT on Top of the Saved d8 Base Model

Use the Kaggle-safe SFT entrypoint here:

- run `scripts.chat_sft_updated`
- load `base:d8_kaggle` from the imported `base_checkpoints`
- use `torchrun --nproc_per_node=2`
- keep `--run=dummy` to avoid wandb login requirements
- use `--device-batch-size=8`
- set `--total-batch-size=16384`
- keep `--mmlu-epochs=1` and `--gsm8k-epochs=1` for a lighter first pass
- disable periodic eval and ChatCORE for this run


In [8]:
# Real d8 SFT on Kaggle 2xT4
# Starts from base:d8_kaggle and saves to chatsft_checkpoints/d8_kaggle
SFT_STEPS = 5000

# ---------------------------------------------------------------------
# Verify base checkpoint is finite before SFT.

base_dir = WORK_CACHE / "base_checkpoints" / "d8_kaggle"
latest_model = sorted(base_dir.glob("model_*.pt"))[-1]
state = torch.load(latest_model, map_location="cpu")
bad = [
    name for name, tensor in state.items()
    if torch.is_floating_point(tensor) and not torch.isfinite(tensor).all()
]
assert not bad, f"Base checkpoint has non-finite tensors: {bad[:8]}"
del state
print("Base checkpoint finite:", latest_model)

# ---------------------------------------------------------------------
# Runtime env

env = os.environ.copy()
env["NANOCHAT_DTYPE"] = "float16"
env["NANOCHAT_DISABLE_COMPILE"] = "1"
env["TORCH_COMPILE_DISABLE"] = "1"

env["NANOCHAT_ADAMW_ALLREDUCE"] = "1"
env["NANOCHAT_SERIAL_OPTIM_COMM"] = "1"

env["OMP_NUM_THREADS"] = "1"
env["PYTHONUNBUFFERED"] = "1"
env["NCCL_P2P_DISABLE"] = "1"
env["NCCL_IB_DISABLE"] = "1"
env["TORCH_NCCL_ASYNC_ERROR_HANDLING"] = "1"
env["TORCH_FR_BUFFER_SIZE"] = "1048576"
env["NCCL_DEBUG"] = "WARN"

env["NANOCHAT_DEBUG_DIST"] = "0"
env["NANOCHAT_DEBUG_STDOUT"] = "0"
env["NANOCHAT_DEBUG_STACK"] = "0"

# ---------------------------------------------------------------------
# Run SFT

cmd = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",
    "-m", "scripts.chat_sft_updated",
    "--",

    "--model-tag=d8_kaggle",
    "--run=dummy",

    "--max-seq-len=1024",
    "--device-batch-size=8",
    "--total-batch-size=16384",

    f"--num-iterations={SFT_STEPS}",
    "--load-optimizer=0",

    "--embedding-lr=0.03",
    "--unembedding-lr=0.0008",
    "--matrix-lr=0.002",
    "--init-lr-frac=0.2",
    "--warmup-ratio=0.05",
    "--warmdown-ratio=0.2",

    "--mmlu-epochs=1",
    "--gsm8k-epochs=1",

    "--eval-every=-1",
    "--eval-tokens=524288",
    "--chatcore-every=-1",
]

print("Running command:")
print(" ".join(cmd))

subprocess.run(
    cmd,
    cwd=WORK_REPO,
    env=env,
    check=True,
)


Base checkpoint finite: /kaggle/working/nanochat_cache/base_checkpoints/d8_kaggle/model_001920.pt
Running command:
torchrun --standalone --nproc_per_node=2 -m scripts.chat_sft_updated -- --model-tag=d8_kaggle --run=dummy --max-seq-len=1024 --device-batch-size=8 --total-batch-size=16384 --num-iterations=5000 --load-optimizer=0 --embedding-lr=0.03 --unembedding-lr=0.0008 --matrix-lr=0.002 --init-lr-frac=0.2 --warmup-ratio=0.05 --warmdown-ratio=0.2 --mmlu-epochs=1 --gsm8k-epochs=1 --eval-every=-1 --eval-tokens=524288 --chatcore-every=-1


[W613 19:45:34.651104579 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
2026-06-13 19:45:44,309 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-13 19:45:44,310 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-13 19:45:44,312 - datasets - INFO - JAX version 0.7.2 available.
2026-06-13 19:45:44,312 - datasets - INFO - JAX version 0.7.2 available.


Autodetected device type: cuda


[W613 19:45:46.401532797 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W613 19:45:46.401778091 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


NCCL version 2.27.5+cuda12.9


...

2026-06-13 19:46:31,350 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/datasets/openai/gsm8k/tree/740312add88f781978c0658806c59bc2815b9866?recursive=false&expand=false "HTTP/1.1 200 OK"
2026-06-13 19:46:31,358 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/HuggingFaceTB/smol-smoltalk/resolve/f73fe857d519ff6ac5af2ea67c4d3834da7b8bcc/smol-smoltalk.py "HTTP/1.1 404 Not Found"
2026-06-13 19:46:31,394 - httpx - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/HuggingFaceTB/smol-smoltalk/HuggingFaceTB/smol-smoltalk.py "HTTP/1.1 404 Not Found"
2026-06-13 19:46:31,418 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/dataset_infos.json "HTTP/1.1 404 Not Found"
2026-06-13 19:46:31,462 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/HuggingFaceTB/smol-smoltalk/resolve/f73fe857d519ff6ac5af2ea67c4d3834da7b8bcc/.huggingfa

step 00001 (0.04%) | loss: 1.306468 | lrm: 0.01 | dt: 1744.30ms | tok/sec: 9,392 | mfu: 0.00 | epoch: 1 | total time: 0.00m
step 00002 (0.06%) | loss: 1.639735 | lrm: 0.01 | dt: 491.22ms | tok/sec: 33,353 | mfu: 0.00 | epoch: 1 | total time: 0.00m
step 00003 (0.08%) | loss: 1.916431 | lrm: 0.02 | dt: 497.86ms | tok/sec: 32,908 | mfu: 0.00 | epoch: 1 | total time: 0.00m
step 00004 (0.10%) | loss: 2.029058 | lrm: 0.02 | dt: 496.47ms | tok/sec: 33,000 | mfu: 0.00 | epoch: 1 | total time: 0.00m
step 00005 (0.12%) | loss: 2.133187 | lrm: 0.02 | dt: 497.74ms | tok/sec: 32,916 | mfu: 0.00 | epoch: 1 | total time: 0.00m
step 00006 (0.14%) | loss: 2.170861 | lrm: 0.03 | dt: 496.06ms | tok/sec: 33,028 | mfu: 0.00 | epoch: 1 | total time: 0.00m
step 00007 (0.16%) | loss: 2.200061 | lrm: 0.03 | dt: 498.23ms | tok/sec: 32,884 | mfu: 0.00 | epoch: 1 | total time: 0.00m
step 00008 (0.18%) | loss: 2.250007 | lrm: 0.04 | dt: 495.78ms | tok/sec: 33,046 | mfu: 0.00 | epoch: 1 | total time: 0.00m
step 000

2026-06-13 20:30:54,871 - nanochat.checkpoint_manager - INFO - Saved model parameters to: /kaggle/working/nanochat_cache/chatsft_checkpoints/d8_kaggle/model_004999.pt
2026-06-13 20:30:54,872 - nanochat.checkpoint_manager - INFO - Saved metadata to: /kaggle/working/nanochat_cache/chatsft_checkpoints/d8_kaggle/meta_004999.json
2026-06-13 20:30:55,556 - nanochat.checkpoint_manager - INFO - Saved optimizer state to: /kaggle/working/nanochat_cache/chatsft_checkpoints/d8_kaggle/optim_004999_rank1.pt
2026-06-13 20:30:56,354 - nanochat.checkpoint_manager - INFO - Saved optimizer state to: /kaggle/working/nanochat_cache/chatsft_checkpoints/d8_kaggle/optim_004999_rank0.pt


Peak memory usage: 6921.56MiB
Total training time: 44.10m
Minimum validation bpb: 0.5304


CompletedProcess(args=['torchrun', '--standalone', '--nproc_per_node=2', '-m', 'scripts.chat_sft_updated', '--', '--model-tag=d8_kaggle', '--run=dummy', '--max-seq-len=1024', '--device-batch-size=8', '--total-batch-size=16384', '--num-iterations=5000', '--load-optimizer=0', '--embedding-lr=0.03', '--unembedding-lr=0.0008', '--matrix-lr=0.002', '--init-lr-frac=0.2', '--warmup-ratio=0.05', '--warmdown-ratio=0.2', '--mmlu-epochs=1', '--gsm8k-epochs=1', '--eval-every=-1', '--eval-tokens=524288', '--chatcore-every=-1'], returncode=0)

## 4. Evaluate the SFT Chat Model

Use a reduced 2-GPU chat evaluation first:

- source is `sft`
- load model tag `d8_kaggle`
- limit to `--max-problems=50`
- use `--batch-size=2` for categorical tasks


In [9]:
env = os.environ.copy()
env["NANOCHAT_DTYPE"] = "float16"
env["NANOCHAT_DISABLE_COMPILE"] = "1"
env["OMP_NUM_THREADS"] = "1"
env["NCCL_P2P_DISABLE"] = "1"
env["NCCL_IB_DISABLE"] = "1"

cmd = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",
    "-m", "scripts.chat_eval",
    "--",
    "-i", "sft",
    "-g", "d8_kaggle",
    "-x", "50",
    "-b", "2",
]

print("Running command:")
print(" ".join(cmd))

subprocess.run(
    cmd,
    cwd=WORK_REPO,
    env=env,
    check=True,
)

Running command:
torchrun --standalone --nproc_per_node=2 -m scripts.chat_eval -- -i sft -g d8_kaggle -x 50 -b 2


[W613 20:31:01.612232982 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
2026-06-13 20:31:05,864 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-13 20:31:05,867 - datasets - INFO - JAX version 0.7.2 available.
2026-06-13 20:31:06,125 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-13 20:31:06,127 - datasets - INFO - JAX version 0.7.2 available.


Autodetected device type: cuda


[W613 20:31:07.382773852 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W613 20:31:07.432421316 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
2026-06-13 20:31:07,472 - nanochat.common - INFO - Distributed world size: 2
2026-06-13 20:31:07,473 - nanochat.checkpoint_manager - INFO - Loading model from /kaggle/working/nanochat_cache/chatsft_checkpoints/d8_kaggle with step 4999
2026-06-13 20:31:07,935 - nanochat.checkpoint_manager - INFO - Building model with config: {'sequence_len': 1024, 'vocab_size': 32768, 'n_layer': 8, 'n_head': 4, 'n_kv_head': 4, 'n_embd': 512, 'window_pattern': 'L'}
2026-06-13 20:31:08,229 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/allenai/ai2_arc/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-13 20:31:08,239 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/allenai/ai2_arc/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"

Final: 15/50 (30.00%)
ARC-Easy accuracy: 30.00%


...

2026-06-13 20:31:13,577 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-13 20:31:13,579 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-13 20:31:13,592 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-06-13 20:31:13,594 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-06-13 20:31:13,653 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/gsm8k.py "HTTP/1.1 404 Not Found"
2026-06-13 20:31:13,657 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k

Rank 0 | 0/25 (0.00%)
Rank 1 | 0/25 (0.00%)
Final: 0/50 (0.00%)
GSM8K accuracy: 0.00%


2026-06-13 20:32:32,782 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/openai_humaneval/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-13 20:32:32,782 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/openai_humaneval/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-13 20:32:32,821 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/openai_humaneval/7dce6050a7d6d172f3cc5c32aa97f52fa1a2e544/README.md "HTTP/1.1 200 OK"
2026-06-13 20:32:32,822 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/openai_humaneval/7dce6050a7d6d172f3cc5c32aa97f52fa1a2e544/README.md "HTTP/1.1 200 OK"
2026-06-13 20:32:32,863 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/openai/openai_humaneval/7dce6050a7d6d172f3cc5c32aa97f52fa1a2e544/README.md "HTTP/1.1 200 OK"
2026-06-13 20:32:32,931 - httpx - INFO - HTTP 

Rank 1 | 0/25 (0.00%)
Rank 0 | 0/25 (0.00%)
Final: 0/50 (0.00%)
HumanEval accuracy: 0.00%
Rank 0 | 23/25 (92.00%))
Rank 1 | 25/25 (100.00%)
Final: 48/50 (96.00%)
SpellingBee accuracy: 96.00%


CompletedProcess(args=['torchrun', '--standalone', '--nproc_per_node=2', '-m', 'scripts.chat_eval', '--', '-i', 'sft', '-g', 'd8_kaggle', '-x', '50', '-b', '2'], returncode=0)

In [10]:
# Optional: save the SFT checkpoint cache as a Kaggle Dataset.
# This stages only the reusable SFT pieces: chatsft_checkpoints and tokenizer.
# Kaggle Dataset id uses a slug, while the title keeps the readable name.
import json

SFT_MODEL_TAG = 'd8_kaggle'
SFT_SOURCE_DIR = WORK_CACHE / 'chatsft_checkpoints' / SFT_MODEL_TAG
TOKENIZER_SOURCE_DIR = WORK_CACHE / 'tokenizer'
SFT_CACHE_DIR = Path('/kaggle/working/nanochat_d8_sft_cache')

DATASET_ID = 'yshuaiqin/nanochat-d8-sft-cache'
TITLE = 'nanochat d8 sft cache'
VERSION_MESSAGE = f'Save {SFT_MODEL_TAG} chat SFT checkpoint cache'

if not DATASET_ID:
    print('Skipping upload. Set DATASET_ID in this cell and rerun it if you want to publish the SFT cache.')
else:
    assert '/' in DATASET_ID, "DATASET_ID should look like '<username>/<dataset-slug>'."
    assert SFT_SOURCE_DIR.exists(), f'Missing SFT checkpoint directory: {SFT_SOURCE_DIR}'
    assert TOKENIZER_SOURCE_DIR.exists(), f'Missing tokenizer directory: {TOKENIZER_SOURCE_DIR}'
    assert sorted(SFT_SOURCE_DIR.glob('model_*.pt')), f'No model_*.pt files found in {SFT_SOURCE_DIR}'
    assert sorted(SFT_SOURCE_DIR.glob('meta_*.json')), f'No meta_*.json files found in {SFT_SOURCE_DIR}'

    if SFT_CACHE_DIR.exists():
        shutil.rmtree(SFT_CACHE_DIR)
    SFT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

    shutil.copytree(WORK_CACHE / 'chatsft_checkpoints', SFT_CACHE_DIR / 'chatsft_checkpoints')
    shutil.copytree(TOKENIZER_SOURCE_DIR, SFT_CACHE_DIR / 'tokenizer')

    identity_path = WORK_CACHE / 'identity_conversations.jsonl'
    if identity_path.exists():
        shutil.copy2(identity_path, SFT_CACHE_DIR / identity_path.name)

    metadata = {
        'title': TITLE,
        'id': DATASET_ID,
        'licenses': [
            {'name': 'CC0-1.0'}
        ]
    }

    metadata_path = SFT_CACHE_DIR / 'dataset-metadata.json'
    metadata_path.write_text(json.dumps(metadata, indent=2))

    print('Dataset metadata:')
    print(metadata_path.read_text())

    print()
    print('Folder size:')
    subprocess.run(['du', '-sh', str(SFT_CACHE_DIR)], check=False)

    print()
    print('Staged files:')
    subprocess.run(['find', str(SFT_CACHE_DIR), '-maxdepth', '3', '-type', 'f'], check=False)

    status = subprocess.run(
        ['kaggle', 'datasets', 'status', DATASET_ID],
        text=True,
        capture_output=True,
    )

    if status.returncode == 0:
        print()
        print(f'Dataset already exists: {DATASET_ID}')
        print('Creating a new dataset version...')
        cmd = [
            'kaggle', 'datasets', 'version',
            '-p', str(SFT_CACHE_DIR),
            '-m', VERSION_MESSAGE,
            '--dir-mode', 'tar',
        ]
    else:
        print()
        print(f'Dataset does not exist yet: {DATASET_ID}')
        print('Creating a new private dataset...')
        cmd = [
            'kaggle', 'datasets', 'create',
            '-p', str(SFT_CACHE_DIR),
            '--dir-mode', 'tar',
        ]

    print()
    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd, text=True)

    if result.returncode != 0:
        raise RuntimeError('Kaggle Dataset upload failed.')

    verify = subprocess.run(
        ['kaggle', 'datasets', 'status', DATASET_ID],
        text=True,
        capture_output=True,
    )

    print()
    print('Post-upload dataset status:')
    if verify.stdout:
        print(verify.stdout)
    if verify.stderr:
        print(verify.stderr)
    assert verify.returncode == 0, 'Kaggle Dataset upload finished, but status verification failed.'

    print()
    print('Done. Verified Kaggle Dataset status successfully.')
    print(f'Dataset URL: https://www.kaggle.com/datasets/{DATASET_ID}')


Dataset metadata:
{
  "title": "nanochat d8 sft cache",
  "id": "yshuaiqin/nanochat-d8-sft-cache",
  "licenses": [
    {
      "name": "CC0-1.0"
    }
  ]
}

Folder size:
2.1G	/kaggle/working/nanochat_d8_sft_cache

Staged files:
/kaggle/working/nanochat_d8_sft_cache/dataset-metadata.json
/kaggle/working/nanochat_d8_sft_cache/identity_conversations.jsonl
/kaggle/working/nanochat_d8_sft_cache/chatsft_checkpoints/d8_kaggle/optim_004999_rank0.pt
/kaggle/working/nanochat_d8_sft_cache/chatsft_checkpoints/d8_kaggle/optim_004999_rank1.pt
/kaggle/working/nanochat_d8_sft_cache/chatsft_checkpoints/d8_kaggle/model_004999.pt
/kaggle/working/nanochat_d8_sft_cache/chatsft_checkpoints/d8_kaggle/meta_004999.json
/kaggle/working/nanochat_d8_sft_cache/tokenizer/tokenizer.pkl
/kaggle/working/nanochat_d8_sft_cache/tokenizer/token_bytes.pt

Dataset does not exist yet: yshuaiqin/nanochat-d8-sft-cache
Creating a new private dataset...

Running: kaggle datasets create -p /kaggle/working/nanochat_d8_sft_cache -

100%|██████████| 2.40M/2.40M [00:00<00:00, 5.55MB/s]


Upload successful: identity_conversations.jsonl (2MB)
Starting upload for file chatsft_checkpoints.tar


100%|██████████| 2.06G/2.06G [00:16<00:00, 136MB/s] 


Upload successful: chatsft_checkpoints.tar (2GB)
Starting upload for file tokenizer.tar


100%|██████████| 540k/540k [00:00<00:00, 1.39MB/s]


Upload successful: tokenizer.tar (540KB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/yshuaiqin/nanochat-d8-sft-cache

Post-upload dataset status:
403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetStatus



AssertionError: Kaggle Dataset upload finished, but status verification failed.

## 5. Serve the SFT Chat Model

Run this after `chatsft_checkpoints/d8_kaggle` exists. The first cell starts or restarts the local NanoChat web server from the SFT checkpoint. The second cell provides a small notebook chat loop against that server.


In [14]:
import time
import requests

SFT_CHECKPOINT_DIR = WORK_CACHE / "chatsft_checkpoints"
SFT_MODEL_TAG = "d8_kaggle"
SERVER_PORT = 8000
BASE_URL = f"http://127.0.0.1:{SERVER_PORT}"

model_dir = SFT_CHECKPOINT_DIR / SFT_MODEL_TAG
assert model_dir.exists(), f"Missing SFT checkpoint directory: {model_dir}"
assert sorted(model_dir.glob("model_*.pt")), f"No model_*.pt files found in {model_dir}"
assert sorted(model_dir.glob("meta_*.json")), f"No meta_*.json files found in {model_dir}"

try:
    if server.poll() is None:
        print("Stopping existing NanoChat server...")
        server.terminate()
        server.wait(timeout=10)
        print("Stopped existing server.")
except NameError:
    pass
except Exception as exc:
    print("Could not stop existing server cleanly:", exc)
    try:
        server.kill()
        server.wait(timeout=10)
        print("Killed existing server.")
    except Exception:
        pass

messages = []

server_env = os.environ.copy()
server_env["NANOCHAT_DTYPE"] = "float16"
server_env["NANOCHAT_DISABLE_COMPILE"] = "1"
server_env["TORCH_COMPILE_DISABLE"] = "1"
server_env["OMP_NUM_THREADS"] = "1"

cmd = [
    sys.executable,
    "-m", "scripts.chat_web",
    "--checkpoint-dir", str(SFT_CHECKPOINT_DIR),
    "--model-tag", SFT_MODEL_TAG,
    "--num-gpus", "1",
    "--port", str(SERVER_PORT),
]

print("Starting NanoChat server:")
print(" ".join(cmd))
server = subprocess.Popen(cmd, cwd=WORK_REPO, env=server_env)
print(f"Server process started with PID {server.pid}.")

SERVER_READY = False
for _ in range(60):
    if server.poll() is not None:
        raise RuntimeError(f"NanoChat server exited early with code {server.returncode}")
    try:
        response = requests.get(f"{BASE_URL}/health", timeout=2)
        if response.ok:
            SERVER_READY = True
            break
    except requests.RequestException:
        pass
    time.sleep(2)

if SERVER_READY:
    print(f"NanoChat server is ready: {BASE_URL}")
else:
    print(f"NanoChat server is still loading. Wait a bit, then use: {BASE_URL}")


Starting NanoChat server:
/usr/bin/python3 -m scripts.chat_web --checkpoint-dir /kaggle/working/nanochat_cache/chatsft_checkpoints --model-tag d8_kaggle --num-gpus 1 --port 8000
Server process started with PID 907.
Autodetected device type: cuda


2026-06-13 20:50:09,390 - nanochat.common - INFO - Distributed world size: 1
INFO:     Started server process [907]
INFO:     Waiting for application startup.
2026-06-13 20:50:09,462 - nanochat.checkpoint_manager - INFO - Loading model from /kaggle/working/nanochat_cache/chatsft_checkpoints/d8_kaggle with step 4999
2026-06-13 20:50:10,089 - nanochat.checkpoint_manager - INFO - Building model with config: {'sequence_len': 1024, 'vocab_size': 32768, 'n_layer': 8, 'n_head': 4, 'n_kv_head': 4, 'n_embd': 512, 'window_pattern': 'L'}
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Starting NanoChat Web Server
Temperature: 0.8, Top-k: 50, Max tokens: 512
Loading nanochat models across GPUs...
Initializing worker pool with 1 GPUs...
Loading model on GPU 0...
All 1 workers initialized!
Server ready at http://localhost:8000
INFO:     127.0.0.1:34914 - "GET /health HTTP/1.1" 200 OK
NanoChat server is ready: http://127.0.0.1:8000


In [15]:
import json
import requests

BASE = globals().get("BASE_URL", "http://127.0.0.1:8000")
messages = []

def ask(prompt, temperature=0.8, top_k=50, max_tokens=512):
    messages.append({"role": "user", "content": prompt})

    payload = {
        "messages": messages,
        "temperature": temperature,
        "top_k": top_k,
        "max_tokens": max_tokens,
    }

    print("Assistant:", end=" ", flush=True)
    answer = ""

    with requests.post(f"{BASE}/chat/completions", json=payload, stream=True, timeout=300) as response:
        response.raise_for_status()
        for line in response.iter_lines(decode_unicode=True):
            if not line or not line.startswith("data: "):
                continue

            data = json.loads(line[len("data: "):])
            if data.get("done"):
                break

            token = data.get("token", "")
            answer += token
            print(token, end="", flush=True)

    print()
    messages.append({"role": "assistant", "content": answer})
    return answer

print(f"Chatting with {BASE}. Type q, quit, or exit to stop. Type reset to clear history.")
while True:
    prompt = input("\nYou: ")
    command = prompt.strip().lower()
    if command in {"q", "quit", "exit"}:
        break
    if command == "reset":
        messages.clear()
        print("Chat history cleared.")
        continue
    ask(prompt)


Chatting with http://127.0.0.1:8000. Type q, quit, or exit to stop. Type reset to clear history.



You:  What's the capital of China?


Assistant: INFO:     127.0.0.1:38550 - "POST /chat/completions HTTP/1.1" 200 OK


2026-06-13 20:50:25,957 - __main__ - INFO - ====================
2026-06-13 20:50:25,957 - __main__ - INFO - [USER]: What's the capital of China?
2026-06-13 20:50:25,957 - __main__ - INFO - --------------------


Chenan is the capital of China. It is a central metropolis in China, and it is the capital of many major cities, such as Guangzhou, Guangxi, and Yingming China. The city is famous for its rich cultural heritage, architecture, and food culture, with many festivals and events throughout its history.

Chenan is also known for its vibrant nightlife scene, with some popular festivals and festivals in the city. The city also offers a wide range of food options, including street food, table lamps, and specialty food shops.

Overall, the city of China is a vibrant and dynamic city that offers a unique blend of traditional Chinese culture, innovative culture, modernity, and a diverse range of offerings.

...

Assistant: INFO:     127.0.0.1:60934 - "POST /chat/completions HTTP/1.1" 200 OK


2026-06-13 20:51:06,848 - __main__ - INFO - ====================
2026-06-13 20:51:06,848 - __main__ - INFO - [USER]: What's the capital of China?
2026-06-13 20:51:06,848 - __main__ - INFO - [ASSISTANT]: Chenan is the capital of China. It is a central metropolis in China, and it is the capital of many major cities, such as Guangzhou, Guangxi, and Yingming China. The city is famous for its rich cultural heritage, architecture, and food culture, with many festivals and events throughout its history.

Chenan is also known for its vibrant nightlife scene, with some popular festivals and festivals in the city. The city also offers a wide range of food options, including street food, table lamps, and specialty food shops.

Overall, the city of China is a vibrant and dynamic city that offers a unique blend of traditional Chinese culture, innovative culture, modernity, and a diverse range of offerings.
2026-06-13 20:51:06,848 - __main__ - INFO - [USER]: What about Beijing?
2026-06-13 20:51:06,8

 Beijing is a major city in China, and its capital city is China. It is a central metropolis in China, with many iconic landmarks and landmarks along the Seine. The city is also famous for its Chinese dining scene, which is a traditional Chinese restaurant in China's Jianin city.

Chenan also offers a wide variety of food options, including street food, table lamps, and specialty food shops. Some of the city's top restaurants and cafes have been known to offer food at Chinese restaurants, including Chinese restaurants and cafes.

In China, the city is also known for its multiculturalism, with many multicultural cities in China including Chenan, Chinese Academy, and Chinese Food Festival. Furthermore, China has a large population of multicultural cities, with some cities serving traditional Chinese cuisine and others serving Chinese cuisine.

In contrast, China has a strong culture and a rich history, with its rich cultural heritage, architecture, and food traditions. The city is also f

2026-06-13 20:51:08,836 - __main__ - INFO - [ASSISTANT] (GPU 0):  Beijing is a major city in China, and its capital city is China. It is a central metropolis in China, with many iconic landmarks and landmarks along the Seine. The city is also famous for its Chinese dining scene, which is a traditional Chinese restaurant in China's Jianin city.

Chenan also offers a wide variety of food options, including street food, table lamps, and specialty food shops. Some of the city's top restaurants and cafes have been known to offer food at Chinese restaurants, including Chinese restaurants and cafes.

In China, the city is also known for its multiculturalism, with many multicultural cities in China including Chenan, Chinese Academy, and Chinese Food Festival. Furthermore, China has a large population of multicultural cities, with some cities serving traditional Chinese cuisine and others serving Chinese cuisine.

In contrast, China has a strong culture and a rich history, with its rich cultura


You:  q


In [16]:
# Stop the NanoChat web server started by the serving cell.
import os
import time

SERVER_PORT = globals().get("SERVER_PORT", 8000)
stopped_any = False

server_process = globals().get("server")
if server_process is not None:
    if server_process.poll() is None:
        print(f"Stopping NanoChat server process {server_process.pid}...")
        server_process.terminate()
        try:
            server_process.wait(timeout=10)
            print("NanoChat server stopped.")
        except subprocess.TimeoutExpired:
            print("Server did not stop after terminate(); killing it...")
            server_process.kill()
            server_process.wait(timeout=10)
            print("NanoChat server killed.")
        stopped_any = True
    else:
        print(f"NanoChat server process already exited with code {server_process.returncode}.")
else:
    print("No notebook `server` variable found.")

if stopped_any:
    time.sleep(2)
    print("Server shutdown complete.")
else:
    print("No running NanoChat server process found.")

Stopping NanoChat server process 907...


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [907]


NanoChat server stopped.
Server shutdown complete.
